# PicMap v2 — Phase 1: Extract Metadata + Sample Photos

This notebook extracts GPS/timestamp metadata from your Takeout album and fetches 3 sample photos per stop.

**Run this once** to generate:
- `trip_metadata.json` — All photo GPS/timestamps for local iteration
- `data.json` — Stop/route data for the frontend
- `output/photos/` — ~100-300 sample photos (3 per stop)

**Phase 2 (local):** Edit `config.json` clustering thresholds and run `build_from_metadata.py` locally — no Drive needed.

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 2: Clone / pull latest code from GitHub ────────────────────────
import os

REPO_URL = 'https://github.com/cmprice1/picmap.git'
BRANCH   = 'v2-colab'
REPO_DIR = '/content/picmap'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    print('Cloning repo...')
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

!echo "Latest commit: $(git -C {REPO_DIR} log -1 --format='%h %s')"

In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────
!pip install -q -r {REPO_DIR}/requirements.txt
print('Dependencies ready.')

In [ ]:
# ── Cell 4: Run Phase 1 — Extract metadata + sample photos ──────────────
# This:
#   1. Copies just .json sidecars to local (fast, ~24MB)
#   2. Parses all GPS/timestamps
#   3. Clusters with loose thresholds (more stops = better coverage)
#   4. Fetches 3 sample photos per stop (~100-300 photos total)
#   5. Outputs trip_metadata.json + data.json

ALBUM = '/content/drive/My Drive/PicMap-V2 Project/Takeout/Google Photos/2025 Great American Roadtrip (iphone)'
OUTPUT = f'{REPO_DIR}/output'
CONFIG = f'{REPO_DIR}/config_phase1.json'
LOCAL_CACHE = '/content/local_sidecars'

!python {REPO_DIR}/extract_metadata.py \
    --album "{ALBUM}" \
    --output "{OUTPUT}" \
    --config "{CONFIG}" \
    --local-cache "{LOCAL_CACHE}"

In [ ]:
# ── Cell 5: Push results to GitHub ──────────────────────────────────────
!git -C {REPO_DIR} config user.email "colab@picmap.dev"
!git -C {REPO_DIR} config user.name "Colab Runner"

# Stage metadata, data.json, and sample photos
!git -C {REPO_DIR} add output/trip_metadata.json output/data.json output/photos/
!git -C {REPO_DIR} commit -m "Phase 1: Extract metadata + sample photos" || echo "Nothing to commit."
!git -C {REPO_DIR} push origin {BRANCH}

print("\n✓ Pushed to GitHub")

In [ ]:
# ── Cell 6: Preview results ─────────────────────────────────────────────
import json
import os

with open(f'{REPO_DIR}/output/data.json') as f:
    data = json.load(f)

with open(f'{REPO_DIR}/output/trip_metadata.json') as f:
    metadata = json.load(f)

photos_dir = f'{REPO_DIR}/output/photos'
num_photos = len([f for f in os.listdir(photos_dir) if os.path.isfile(os.path.join(photos_dir, f))])

print(f"Trip: {data['trip']['title']}")
print(f"Date range: {data['trip']['start_date']} to {data['trip']['end_date']}")
print(f"\nMetadata: {len(metadata['photos'])} total photos with GPS/timestamps")
print(f"Sample photos downloaded: {num_photos}")
print(f"\nStops: {len(data['stops'])}")
print(f"Waypoints: {len(data['waypoints'])}")
print(f"\n{'='*60}")
print("Stops breakdown:")
for s in data['stops']:
    print(f"  {s['order']:2d}. {s['name'][:30]:<30} ({s['type']}, {s['photo_count']} photos, {len(s['photos'])} samples)")

---

## Next Steps (Phase 2 — Local)

1. **Pull to your local machine:**
   ```bash
   git pull origin v2-colab
   ```

2. **Iterate on clustering locally:**
   - Edit `config.json` thresholds (`cluster_radius_km`, `cluster_time_gap_hours`)
   - Run: `python build_from_metadata.py --config config.json`
   - Test: `python -m http.server 8080 --directory output`

3. **When happy with stops, deploy to Netlify**